In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

BASE_URL = "https://www.419scam.org/emails/"
TARGET_MONTHS = ["2024-12", "2024-11"]

def get_index_links():
    """Scrapes the main page to collect only links leading to individual date index pages for target months."""
    response = requests.get(BASE_URL)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    index_links = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if any(month in href for month in TARGET_MONTHS) and href.endswith('index.htm'):
            index_links.append(BASE_URL + href)
    
    return index_links

In [26]:
def get_email_links(index_url):
    """Scrapes an index page to collect only email links that match the pattern and their texts."""
    response = requests.get(index_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    email_links = []
    date_part = "/".join(index_url.split('/')[-3:-1])  # Extract YYYY-MM/DD from URL
    base_path = index_url.rsplit('/', 1)[0] + '/'
    
    for link in soup.find_all('a', href=True):
        href = link['href']
        if re.match(r"""\d+\.\d+\.htm$""", href):
            email_links.append((link.text.strip(), base_path + href, date_part))
    
    return email_links

In [ ]:
def main():
    index_links = get_index_links()
    
    all_data = []
    for index_url in index_links:
        email_links = get_email_links(index_url)
        for text, url, date in email_links:
            all_data.append([text, url, date])
        
        time.sleep(2)
    
    df = pd.DataFrame(all_data, columns=["Link Text", "URL", "Date"])
    
    # Save results
    df.to_csv("419_scam_emails.csv", index=False)

if __name__ == "__main__":
    main()    
  

In [49]:
df = pd.read_csv("419_scam_emails.csv")
df.head(5)

,Link Text,URL,Date
0,HSBC Funds Payment,https://www.419scam.org/emails/2024-12/01/0450...,2024-12/01
1,Hello,https://www.419scam.org/emails/2024-12/01/0450...,2024-12/01
2,You Must Not Ignore This Message,https://www.419scam.org/emails/2024-12/01/0450...,2024-12/01
3,(no subject),https://www.419scam.org/emails/2024-12/01/0450...,2024-12/01
4,Re: Re: Re:,https://www.419scam.org/emails/2024-12/01/0450...,2024-12/01
